# Notebook 07: Figure 4 Replication (Residual Hubble Diagram)

In this notebook, we plot the residuals of Supernovae and BAO data relative to a standard $\Lambda$CDM model to visualize the tension and its resolution after correction.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from astropy.cosmology import FlatLambdaCDM

# 1. Setup Cosmology/Baseline
cosmo = FlatLambdaCDM(H0=73.04, Om0=0.334) # Reference LCDM

# 2. Load Data
path_sn_orig = r'../data/external/PantheonPlus/Pantheon+_Data/4_DISTANCES_AND_COVAR/Pantheon+SH0ES.dat'
path_sn_corr = r'../data/external/PantheonPlus_Corrected/Pantheon+_Data/4_DISTANCES_AND_COVAR/Pantheon+SH0ES.dat'
bao_mean_file = r'../data/external/DESI_BAO/desi_2024_gaussian_bao_ALL_GCcomb_mean.txt'

sn_orig = pd.read_csv(path_sn_orig, sep='\s+', skiprows=0 if 'CID' in open(path_sn_orig).read(100) else 1)
sn_corr = pd.read_csv(path_sn_corr, sep='\s+', skiprows=0 if 'CID' in open(path_sn_corr).read(100) else 1)
bao_df = pd.read_csv(bao_mean_file, delim_whitespace=True, comment='#', names=['z', 'value', 'type'])

# 3. Calculate SN Residuals
model_sn = cosmo.distmod(sn_orig['zHD'].values).value
resid_orig = sn_orig['MU_SH0ES'] - model_sn
resid_corr = sn_corr['MU_SH0ES'] - model_sn

# 4. Calculate BAO Residuals
# BAO is in units of DM/rs, DH/rs, DV/rs.
# We need mu_bao = 5*log10(dL_bao) + 25.
# dL(z) = (1+z) * DM(z).
# DM_bao = (DM/rs)_data * rs_theory
rd_theory = 147.09 # Mpc (Placeholder for standard cosmological rs)

bao_mu = []
for idx, row in bao_df.iterrows():
    if row['type'] == 'DM_over_rs':
        DM = row['value'] * rd_theory
        dL = DM * (1 + row['z'])
        mu = 5 * np.log10(dL) + 25
        bao_mu.append({'z': row['z'], 'mu': mu})
    elif row['type'] == 'DV_over_rs':
        # Complex conversion if exact needed, using DM as proxy for scaling
        pass

bao_mu_df = pd.DataFrame(bao_mu)
model_bao = cosmo.distmod(bao_mu_df['z'].values).value
resid_bao = bao_mu_df['mu'] - model_bao

# 5. Plot
plt.figure(figsize=(10, 6))
plt.axhline(0, color='k', linestyle='--', label='Flat $\Lambda$CDM (Baseline)')

# SN Data (Binned for clarity)
bins = np.logspace(-2, 0.2, 12)
for df_loop, label, color in zip([sn_orig, sn_corr], ['Pantheon+ (Orig)', 'Pantheon+ (Corrected)'], ['gray', 'red']):
    resid = df_loop['MU_SH0ES'] - cosmo.distmod(df_loop['zHD'].values).value
    c = pd.cut(df_loop['zHD'], bins)
    m = df_loop.groupby(c, observed=False)[['zHD']].mean()
    v = pd.Series(resid).groupby(c, observed=False).mean()
    plt.plot(m['zHD'], v, 'o-', color=color, label=label)

# BAO Data
plt.errorbar(bao_mu_df['z'], resid_bao, yerr=0.03, fmt='s', color='blue', label='DESI BAO (DM/rs)')

plt.xscale('log')
plt.xlabel('Redshift z')
plt.ylabel('Residual $\Delta \mu$ (mag)')
plt.title('Figure 4 Replication: SN-BAO (Dis)agreement')
plt.legend()
plt.grid(True, which='both', alpha=0.3)
plt.savefig('fig4_residual_agreement.png')
plt.show()